# 🚀 Notebook 9: Accuracy Boost (Target: 65%+)
## Three Targeted Improvements — No Graph Reconstruction Required

**Why previous models plateaued at ~59%:**
1. Node features (6) miss per-node FC context (how connected each ROI is)
2. No residual connections → gradient flow issues in 3-layer GAT
3. Graph-level decision ignores global FC statistics
4. Single model is noisy — ensemble smooths decision boundaries

| Improvement | Technique | Estimated Gain |
|---|---|---|
| **8-feature nodes** | Mean + std edge weight per node (from existing edge_attr) | +2–4% |
| **Residual GAT + Global Feats** | Skip connections + 5 graph stats in classifier head | +2–4% |
| **3-Seed Ensemble** | Average softmax probabilities across 3 seeds | +1–3% |

> ⚡ **Runtime:** ~45–60 min on GPU (3 models × 300 epochs)

In [1]:
# ─── CELL 1: Install ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)
import torch
cuda_tag = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
pyg_url  = f'https://data.pyg.org/whl/torch-{torch.__version__.split("+")[0]}+{cuda_tag}.html'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch_scatter', 'torch_sparse', 'torch_cluster', '-f', pyg_url
], check=False)
print(f'✅ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

✅ PyTorch 2.11.0+cu128 | CUDA: True


In [2]:
# ─── CELL 2: Imports ──────────────────────────────────────────────────────────
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d
from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from torch_geometric.utils import dropout_edge
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from google.colab import drive

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Imports done | Device: {DEVICE}')

✅ Imports done | Device: cuda


In [3]:
# ─── CELL 3: Mount Drive & Paths ──────────────────────────────────────────────
drive.mount('/content/drive')
BASE_DIR   = '/content/drive/MyDrive/ASD_GNN_Research2'
GRAPHS_DIR = os.path.join(BASE_DIR, 'graphs')
MODELS_DIR = os.path.join(BASE_DIR, 'models', 'GAT')
METRICS_DIR= os.path.join(BASE_DIR, 'results', 'metrics')
FIGURES_DIR= os.path.join(BASE_DIR, 'results', 'figures')
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


In [4]:
# ─── CELL 4: On-the-fly Feature Augmentation (6 → 8 per node) ────────────────
#
# Existing .pt files already contain edge_attr (Pearson correlation values).
# We compute 2 extra features per node:
#   Feature 7: mean |edge_attr| for all edges incident to node i
#              = How strongly connected this ROI is on average
#   Feature 8: std  |edge_attr| for all edges incident to node i
#              = How variable its connectivity strength is
# NO graph re-construction needed — computed on-the-fly at load time.

def augment_features(data):
    n_nodes = data.x.size(0)
    if data.edge_attr is not None and data.edge_index.size(1) > 0:
        ea  = data.edge_attr.squeeze(-1).abs()        # [E]
        src = data.edge_index[0]                       # [E]
        dst = data.edge_index[1]                       # [E]
        all_nodes = torch.cat([src, dst])              # [2E]
        all_ea    = torch.cat([ea,  ea ])              # [2E]
        node_mean = torch.zeros(n_nodes)
        node_std  = torch.zeros(n_nodes)
        for v in range(n_nodes):
            mask = (all_nodes == v)
            if mask.sum() > 0:
                vals = all_ea[mask]
                node_mean[v] = vals.mean()
                node_std[v]  = vals.std() if vals.numel() > 1 else 0.0
        extra = torch.stack([node_mean, node_std], dim=1)
    else:
        extra = torch.zeros(n_nodes, 2)
    data.x = torch.cat([data.x, extra], dim=1)        # [N, 6] → [N, 8]
    return data


def compute_global_feats(data):
    """5 graph-level summary statistics for each subject."""
    n_nodes = data.x.size(0)
    n_edges = data.edge_index.size(1)
    if data.edge_attr is not None and n_edges > 0:
        ea = data.edge_attr.squeeze(-1).abs()
        f  = [ea.mean().item(),
               ea.std().item(),
               n_edges / (n_nodes * (n_nodes - 1)),   # graph density
               n_edges / n_nodes,                     # mean degree
               ea.max().item()]                        # peak FC strength
    else:
        f = [0.0] * 5
    return torch.tensor([f], dtype=torch.float)        # shape [1, 5] for PyG batching


print('✅ Augmentation functions defined')
print('   Node: 6 → 8 features (+ mean & std edge weight per node)')
print('   Graph-level: 5 global statistics injected into classifier head')

✅ Augmentation functions defined
   Node: 6 → 8 features (+ mean & std edge weight per node)
   Graph-level: 5 global statistics injected into classifier head


In [5]:
# ─── CELL 5: Augmented Dataset & Verification ─────────────────────────────────

class AugmentedGraphDataset(Dataset):
    def __init__(self, folder_path):
        super().__init__()
        self.folder_path = folder_path
        self.file_names  = sorted([f for f in os.listdir(folder_path) if f.endswith('.pt')])

    def len(self): return len(self.file_names)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.folder_path, self.file_names[idx]),
            weights_only=False
        )
        data = augment_features(data)
        data.global_feats = compute_global_feats(data)
        return data

# Verify
sample = AugmentedGraphDataset(os.path.join(GRAPHS_DIR, 'train')).get(0)
print(f'✅ Node feature shape:  {sample.x.shape}   (expect [116, 8])')
print(f'   Global feats shape: {sample.global_feats.shape} (expect [5])')
print(f'   Global feats:       {sample.global_feats.numpy().round(4)}')

✅ Node feature shape:  torch.Size([116, 8])   (expect [116, 8])
   Global feats shape: torch.Size([1, 5]) (expect [5])
   Global feats:       [[ 0.4742  0.119   0.5535 63.6552  0.9098]]


In [6]:
# ─── CELL 6: ResGATClassifier Architecture ───────────────────────────────────
#
# Improvements over Notebook 07 ImprovedGATClassifier:
#   1. Input projection layer: 8 → 256 (same dim as GAT output so residuals work)
#   2. Residual connections at each conv layer: x_out = conv(x) + x
#   3. Global feature injection: classifier head receives [pool(128) | global(5)] = 133

class ResGATClassifier(torch.nn.Module):
    def __init__(self, num_node_features=8, hidden=64,
                 num_classes=2, heads=4, dropout=0.3, n_global_feats=5):
        super().__init__()
        self.dropout = dropout
        dim = hidden * heads  # 256

        self.input_proj = Linear(num_node_features, dim)
        self.input_bn   = BatchNorm1d(dim)

        self.conv1 = GATConv(dim, hidden, heads=heads, concat=True, dropout=0.2)
        self.bn1   = BatchNorm1d(dim)

        self.conv2 = GATConv(dim, hidden, heads=heads, concat=True, dropout=0.2)
        self.bn2   = BatchNorm1d(dim)

        self.conv3 = GATConv(dim, hidden, heads=1, concat=False, dropout=0.2)
        self.bn3   = BatchNorm1d(hidden)

        in_cls    = hidden * 2 + n_global_feats   # 128 + 5 = 133
        self.lin1 = Linear(in_cls, hidden)
        self.bn4  = BatchNorm1d(hidden)
        self.lin2 = Linear(hidden, hidden // 2)
        self.lin3 = Linear(hidden // 2, num_classes)

    def forward(self, x, edge_index, batch, edge_attr=None, global_feats=None):
        x  = F.elu(self.input_bn(self.input_proj(x)))          # [N, 256]

        x1 = F.elu(self.bn1(self.conv1(x,  edge_index, edge_attr=edge_attr)))
        x1 = F.dropout(x1, p=self.dropout, training=self.training)
        x1 = x1 + x                                            # Residual

        x2 = F.elu(self.bn2(self.conv2(x1, edge_index, edge_attr=edge_attr)))
        x2 = F.dropout(x2, p=self.dropout, training=self.training)
        x2 = x2 + x1                                           # Residual

        x3 = F.elu(self.bn3(self.conv3(x2, edge_index, edge_attr=edge_attr)))

        xg = torch.cat([global_mean_pool(x3, batch),
                         global_max_pool(x3, batch)], dim=1)   # [B, 128]

        if global_feats is not None:
            if global_feats.dim() == 1:
                global_feats = global_feats.view(-1, 5)
            xg = torch.cat([xg, global_feats], dim=1)          # [B, 133]

        xg = F.elu(self.bn4(self.lin1(xg)))
        xg = F.dropout(xg, p=self.dropout, training=self.training)
        xg = F.elu(self.lin2(xg))
        return self.lin3(xg)


n_params = sum(p.numel() for p in ResGATClassifier().parameters())
print(f'✅ ResGATClassifier: {n_params:,} parameters')

✅ ResGATClassifier: 164,002 parameters


In [7]:
# ─── CELL 7: Data Loading ─────────────────────────────────────────────────────
BATCH_SIZE = 64

train_dataset = AugmentedGraphDataset(os.path.join(GRAPHS_DIR, 'train'))
val_dataset   = AugmentedGraphDataset(os.path.join(GRAPHS_DIR, 'val'))
test_dataset  = AugmentedGraphDataset(os.path.join(GRAPHS_DIR, 'test'))

# Combine train+val, carve 15% as internal validation for early stopping
trainval_list = list(train_dataset) + list(val_dataset)
trainval_y    = np.array([d.y.item() for d in trainval_list])
tr_idx, va_idx= train_test_split(range(len(trainval_list)), test_size=0.15,
                                   stratify=trainval_y, random_state=42)
internal_train = [trainval_list[i] for i in tr_idx]
internal_val   = [trainval_list[i] for i in va_idx]

train_loader = DataLoader(internal_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(internal_val,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,   batch_size=BATCH_SIZE, shuffle=False)

train_y = np.array([d.y.item() for d in internal_train])
n_t = len(train_y); n_asd = (train_y==1).sum(); n_ctrl = (train_y==0).sum()
class_weights = torch.tensor([n_t/(2.*n_ctrl), n_t/(2.*n_asd)], dtype=torch.float)

print(f'Train: {len(internal_train)} | Val: {len(internal_val)} | Test: {len(test_dataset)}')
print(f'ASD: {n_asd} ({100*n_asd/n_t:.1f}%) | Control: {n_ctrl}')
print(f'Class weights: ctrl={class_weights[0]:.3f} | asd={class_weights[1]:.3f}')

Train: 629 | Val: 111 | Test: 131
ASD: 291 (46.3%) | Control: 338
Class weights: ctrl=0.930 | asd=1.081


In [8]:
# ─── CELL 8: Training & Evaluation Functions ──────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, device, drop_edge_p=0.10):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for data in loader:
        data = data.to(device)
        ea   = data.edge_attr.squeeze(-1) if data.edge_attr is not None else None
        gf   = data.global_feats.to(device) if hasattr(data,'global_feats') else None
        if drop_edge_p > 0:
            ei_aug, mask = dropout_edge(data.edge_index, p=drop_edge_p, training=True)
            ea_aug = ea[mask] if ea is not None else None
        else:
            ei_aug, ea_aug = data.edge_index, ea
        optimizer.zero_grad()
        out  = model(data.x, ei_aug, data.batch, ea_aug, gf)
        loss = criterion(out, data.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        correct    += (out.argmax(1) == data.y).sum().item()
        n          += data.num_graphs
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    preds, labels, probs, loss_sum, n = [], [], [], 0.0, 0
    for data in loader:
        data = data.to(device)
        ea   = data.edge_attr.squeeze(-1) if data.edge_attr is not None else None
        gf   = data.global_feats.to(device) if hasattr(data,'global_feats') else None
        out  = model(data.x, data.edge_index, data.batch, ea, gf)
        p    = torch.softmax(out, 1)
        loss_sum += criterion(out, data.y).item() * data.num_graphs
        n        += data.num_graphs
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(data.y.cpu().numpy())
        probs.extend(p[:, 1].cpu().numpy())
    preds  = np.array(preds); labels = np.array(labels); probs = np.array(probs)
    acc    = accuracy_score(labels, preds)
    f1     = f1_score(labels, preds, average='weighted', zero_division=0)
    sens   = float((preds[labels==1]==1).mean()) if (labels==1).sum()>0 else 0.0
    spec   = float((preds[labels==0]==0).mean()) if (labels==0).sum()>0 else 0.0
    return loss_sum/n, acc, f1, sens, spec, preds, labels, probs

print('✅ Train/evaluate functions defined')

✅ Train/evaluate functions defined


In [9]:
# ─── CELL 9: Single-Seed Training Helper ──────────────────────────────────────

def train_single_model(seed, epochs=300, lr=3e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed(seed)

    model     = ResGATClassifier(num_node_features=8, hidden=64,
                                  num_classes=2, heads=4, dropout=0.3, n_global_feats=5).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2, eta_min=1e-5)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=0.05)

    best_f1, best_state, best_ep = 0.0, None, 0
    ckpt = os.path.join(MODELS_DIR, f'resgat_seed{seed}.pt')

    print(f'\n── Seed {seed} ─────────────────────────────────────────────────')
    print(f'  {"Ep":>4} {"TLoss":>7} {"TAcc":>6} {"VF1":>6} {"VAcc":>6} {"Sens":>6}')
    print('  ' + '-'*48)

    start = time.time()
    for ep in range(1, epochs+1):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        _, va, vf, vs, _, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        if vf > best_f1:
            best_f1 = vf; best_ep = ep
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if ep % 50 == 0 or ep == best_ep:
            mk = ' ★' if ep == best_ep else ''
            print(f'  {ep:4d} {tl:7.4f} {ta:6.4f} {vf:6.4f} {va:6.4f} {vs:6.4f}{mk}')

    print(f'  Done in {(time.time()-start)/60:.1f} min | Best F1={best_f1:.4f} @ ep {best_ep}')
    model.load_state_dict(best_state)
    torch.save({'seed': seed, 'epoch': best_ep, 'model_state_dict': model.state_dict(),
                 'val_f1': best_f1, 'architecture': 'ResGATClassifier'}, ckpt)
    print(f'  Saved: {ckpt}')
    return model, best_f1

print('✅ Single-seed trainer defined')

✅ Single-seed trainer defined


In [ ]:
# ─── CELL 10: Train 3-Seed Ensemble ──────────────────────────────────────────
SEEDS = [42, 123, 2024]

trained_models, val_f1s = [], []
for seed in SEEDS:
    m, f1 = train_single_model(seed=seed, epochs=300)
    trained_models.append(m)
    val_f1s.append(f1)

print(f'\n✅ Ensemble training complete!')
for s, f in zip(SEEDS, val_f1s):
    print(f'   Seed {s}: val F1 = {f:.4f}')
print(f'   Mean val F1: {np.mean(val_f1s):.4f}')


── Seed 42 ─────────────────────────────────────────────────
    Ep   TLoss   TAcc    VF1   VAcc   Sens
  ------------------------------------------------
     1  0.7145 0.4757 0.5646 0.5856 0.3529 ★
     2  0.6998 0.5174 0.5947 0.6126 0.3922 ★
    50  0.6798 0.5833 0.5769 0.5766 0.5490
    51  0.6823 0.5503 0.6200 0.6216 0.5490 ★
   100  0.6703 0.6094 0.5310 0.5405 0.7059
   150  0.6579 0.5972 0.5310 0.5405 0.7059
   194  0.6521 0.6215 0.6312 0.6306 0.6275 ★
   200  0.6383 0.6493 0.5574 0.5676 0.7451
   250  0.6271 0.6441 0.5414 0.5495 0.7059
   300  0.6293 0.6493 0.6195 0.6216 0.7255
  Done in 3.5 min | Best F1=0.6312 @ ep 194
  Saved: /content/drive/MyDrive/ASD_GNN_Research2/models/GAT/resgat_seed42.pt

── Seed 123 ─────────────────────────────────────────────────
    Ep   TLoss   TAcc    VF1   VAcc   Sens
  ------------------------------------------------
     1  0.7171 0.5122 0.5210 0.5766 0.2157 ★
     5  0.6922 0.5208 0.5277 0.5856 0.2157 ★
     6  0.6956 0.5469 0.5989 0.6216 0

In [ ]:
# ─── CELL 11: Ensemble Test Set Evaluation ────────────────────────────────────

@torch.no_grad()
def ensemble_predict(models, loader, device):
    for m in models: m.eval()
    all_probs, all_labels = [], []
    for data in loader:
        data = data.to(device)
        ea = data.edge_attr.squeeze(-1) if data.edge_attr is not None else None
        gf = data.global_feats.to(device) if hasattr(data,'global_feats') else None
        batch_probs = [torch.softmax(m(data.x, data.edge_index, data.batch, ea, gf), 1)[:,1]
                        .cpu().numpy() for m in models]
        all_probs.extend(np.mean(batch_probs, axis=0).tolist())
        all_labels.extend(data.y.cpu().numpy())
    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs >= 0.5).astype(int)
    return preds, labels, probs


ens_preds, ens_labels, ens_probs = ensemble_predict(trained_models, test_loader, DEVICE)

ens_acc  = accuracy_score(ens_labels, ens_preds)
ens_f1   = f1_score(ens_labels, ens_preds, average='weighted', zero_division=0)
ens_auc  = roc_auc_score(ens_labels, ens_probs)
ens_cm   = confusion_matrix(ens_labels, ens_preds)
tn, fp, fn, tp = ens_cm.ravel()
ens_sens = tp / (tp + fn + 1e-8)
ens_spec = tn / (tn + fp + 1e-8)

print('=' * 58)
print('  ENSEMBLE TEST SET RESULTS (3-Seed ResGAT)')
print('=' * 58)
print(f'  Accuracy:    {ens_acc:.4f}  ({100*ens_acc:.2f}%)')
print(f'  Weighted F1: {ens_f1:.4f}')
print(f'  AUC-ROC:     {ens_auc:.4f}')
print(f'  Sensitivity: {ens_sens:.4f}  (ASD Recall)')
print(f'  Specificity: {ens_spec:.4f}  (Control Recall)')
print(f'  CM → TP={tp} TN={tn} FP={fp} FN={fn}')
print('=' * 58)

In [ ]:
# ─── CELL 12: 5-Fold Stratified Cross-Validation ─────────────────────────────
all_graphs = list(train_dataset) + list(val_dataset) + list(test_dataset)
all_y      = np.array([d.y.item() for d in all_graphs])
skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

print('5-Fold Stratified Cross-Validation (200 epochs per fold)...')

for fold, (tr_i, va_i) in enumerate(skf.split(np.zeros(len(all_graphs)), all_y)):
    print(f'─── Fold {fold+1}/5 ──────────────────────────────────────')
    ft = [all_graphs[i] for i in tr_i]
    fv = [all_graphs[i] for i in va_i]
    fy = np.array([d.y.item() for d in ft])
    nf = len(fy)
    fcw= torch.tensor([nf/(2.*(fy==0).sum()), nf/(2.*(fy==1).sum())], dtype=torch.float)

    fl  = DataLoader(ft, batch_size=64, shuffle=True, drop_last=True)
    fvl = DataLoader(fv, batch_size=64, shuffle=False)

    torch.manual_seed(42)
    fm  = ResGATClassifier(num_node_features=8, hidden=64, num_classes=2, heads=4, dropout=0.3, n_global_feats=5).to(DEVICE)
    fo  = torch.optim.AdamW(fm.parameters(), lr=3e-4, weight_decay=1e-4)
    fs  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(fo, T_0=50, T_mult=2, eta_min=1e-5)
    fc  = torch.nn.CrossEntropyLoss(weight=fcw.to(DEVICE), label_smoothing=0.05)

    bf, bs = 0.0, None
    for ep in range(1, 201):
        train_one_epoch(fm, fl, fo, fc, DEVICE)
        fs.step()
        _, _, vf, _, _, _, _, _ = evaluate(fm, fvl, fc, DEVICE)
        if vf > bf: bf = vf; bs = {k: v.clone() for k, v in fm.state_dict().items()}

    fm.load_state_dict(bs)
    _, va, vf, vs, vc, _, fvl_y, vp = evaluate(fm, fvl, fc, DEVICE)
    fv_y = np.array([d.y.item() for d in fv])
    try: vauc = roc_auc_score(fv_y, vp)
    except: vauc = 0.5
    cv_results.append({'fold': fold+1, 'accuracy': va, 'f1': vf, 'auc': vauc, 'sensitivity': vs, 'specificity': vc})
    print(f'   Acc={va:.4f} F1={vf:.4f} AUC={vauc:.4f} Sens={vs:.4f} Spec={vc:.4f}')

cv_df = pd.DataFrame(cv_results)
print('\n' + '='*55)
print('  5-FOLD CV MEAN ± STD (ResGAT)')
print('='*55)
for col in ['accuracy','f1','auc','sensitivity','specificity']:
    m, s = cv_df[col].mean(), cv_df[col].std()
    print(f'  {col:<14}: {m:.4f} ± {s:.4f}')
cv_df.to_csv(os.path.join(METRICS_DIR, 'resgat_cv_results.csv'), index=False)
print(f'\n✅ Saved: results/metrics/resgat_cv_results.csv')

In [ ]:
# ─── CELL 13: Visualization ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'ResGAT Ensemble — Test Set N={len(ens_labels)}', fontsize=13, fontweight='bold')

ax = axes[0]
ax.imshow(ens_cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        pct = ens_cm[i,j] / ens_cm[i,:].sum()
        ax.text(j, i, f'{ens_cm[i,j]}\n({100*pct:.1f}%)', ha='center', va='center',
                 fontsize=13, color='white' if pct > 0.5 else 'black', fontweight='bold')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Control','ASD']); ax.set_yticklabels(['Control','ASD'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Ensemble)', fontweight='bold')

ax2 = axes[1]
fpr, tpr, _ = roc_curve(ens_labels, ens_probs)
ax2.plot(fpr, tpr, 'b-', lw=2.5, label=f'ResGAT Ensemble (AUC={ens_auc:.3f})')
ax2.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random')
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.set_title('ROC Curve (Ensemble)', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
sp = os.path.join(FIGURES_DIR, 'resgat_ensemble_results.png')
plt.savefig(sp, dpi=200, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {sp}')

In [ ]:
# ─── CELL 14: Final Summary & Comparison ──────────────────────────────────────
prev = {'SVM': (0.542,0.389,0.467,0.016), 'Baseline GAT': (0.527,0.496,0.565,0.262),
         'Improved GAT NB07': (0.592,0.580,0.567,0.578)}

print('='*70)
print('  FULL MODEL COMPARISON')
print('='*70)
print(f'  {"Model":<26} {"Accuracy":>9} {"F1":>8} {"AUC":>8} {"Sensitivity":>12}')
print('-'*70)
for name,(a,f,u,s) in prev.items():
    print(f'  {name:<26} {a:>9.4f} {f:>8.4f} {u:>8.4f} {s:>12.4f}')
print('-'*70)
print(f'  {"► ResGAT Ensemble (NB10)":<26} {ens_acc:>9.4f} {ens_f1:>8.4f} {ens_auc:>8.4f} {ens_sens:>12.4f}')
print('='*70)
print(f'\n  CV Results (ResGAT vs NB07):')
print(f'  Accuracy:  0.5917 → {cv_df["accuracy"].mean():.4f}  ({cv_df["accuracy"].mean()-0.5917:+.4f})')
print(f'  F1:        0.5803 → {cv_df["f1"].mean():.4f}  ({cv_df["f1"].mean()-0.5803:+.4f})')
print(f'  AUC:       0.5672 → {cv_df["auc"].mean():.4f}  ({cv_df["auc"].mean()-0.5672:+.4f})')
print(f'\n✅ Notebook 10 complete! Next: re-run NB06 XAI with resgat_seed42.pt checkpoint.')